# 01 — Document Ingestion

This notebook performs **Phase 02 only** for the official OpenStax *Introduction to Business* PDF. It uses **PyMuPDF** to extract text page-by-page, identifies common extraction problems, applies conservative cleanup, preserves book context, and writes structured page records to `data/processed/`.

> **Scope boundary:** This notebook does not create embeddings, a vector database, retrieval logic, reranking, LLM calls, prompts, LangGraph graphs, or any other RAG component. OpenStax permission must be confirmed before any future AI ingestion activity.

In [1]:
from __future__ import annotations

import json
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF
from IPython.display import JSON, Markdown, display

PDF_NAME = 'introduction-to-business-openstax.pdf'
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'data' / 'raw' / PDF_NAME).exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(f'Could not find data/raw/{PDF_NAME} from {Path.cwd()}')

PDF_PATH = PROJECT_ROOT / 'data' / 'raw' / PDF_NAME
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = {
    'title': 'Introduction to Business',
    'publisher': 'OpenStax, Rice University',
    'official_book_url': 'https://openstax.org/details/books/introduction-business',
    'official_pdf_url': 'https://assets.openstax.org/oscms-prodcms/media/documents/IntroductionToBusiness-OP_8D04gAa.pdf',
    'local_filename': str(PDF_PATH.relative_to(PROJECT_ROOT)),
}
print(f'Project root: {PROJECT_ROOT}')
print(f'Source PDF:   {PDF_PATH}')
print(f'Output folder: {PROCESSED_DIR}')

Project root: /home/ubuntu/business-knowledge-ai
Source PDF:   /home/ubuntu/business-knowledge-ai/data/raw/introduction-to-business-openstax.pdf
Output folder: /home/ubuntu/business-knowledge-ai/data/processed


## 1. Load the PDF and inspect document-level metadata

The page count and document metadata provide a basic completeness check before extraction begins.

In [2]:
with fitz.open(PDF_PATH) as document:
    page_count = len(document)
    pdf_metadata = {key: value for key, value in document.metadata.items() if value}

print(f'Page count: {page_count}')
display(JSON({'pdf_path': str(PDF_PATH), 'page_count': page_count, 'metadata': pdf_metadata}))
assert page_count > 0, 'The PDF contains no pages.'

Page count: 744


<IPython.core.display.JSON object>

## 2. Extract text page-by-page

PyMuPDF block extraction is sorted top-to-bottom and left-to-right before blocks are joined. The raw text is retained unchanged alongside the cleaned text for auditability.

In [3]:
def extract_page_text(page: fitz.Page) -> str:
    blocks = page.get_text('blocks', sort=True)
    return '\n'.join(block[4].strip() for block in blocks if block[4].strip()).strip()

with fitz.open(PDF_PATH) as document:
    raw_page_text = {page_number: extract_page_text(page) for page_number, page in enumerate(document, start=1)}

character_counts = {page: len(text) for page, text in raw_page_text.items()}
print(f'Extracted {len(raw_page_text)} page payloads.')
print(f'Characters per page — min: {min(character_counts.values())}, max: {max(character_counts.values())}, total: {sum(character_counts.values()):,}')
assert len(raw_page_text) == page_count

Extracted 744 page payloads.
Characters per page — min: 0, max: 4803, total: 2,086,011


### Raw extraction samples

These samples make the page-by-page result visible without displaying the complete textbook in notebook output.

In [4]:
sample_page_numbers = list(dict.fromkeys([1, min(3, page_count), min(100, page_count), page_count]))
for page_number in sample_page_numbers:
    text = raw_page_text[page_number]
    display(Markdown(f'#### PDF page {page_number} — {len(text):,} extracted characters'))
    print(text[:1500] if text else '[No extractable text on this page.]')
    print('\n' + '—' * 90 + '\n')

#### PDF page 1 — 0 extracted characters

[No extractable text on this page.]

——————————————————————————————————————————————————————————————————————————————————————————



#### PDF page 3 — 374 extracted characters

Introduction to Business
SENIOR CONTRIBUTING AUTHORS 
LAWRENCE J. GITMAN, SAN DIEGO STATE UNIVERSITY - EMERITUS 
CARL MCDANIEL, UNIVERSITY OF TEXAS, ARLINGTON 
AMIT SHAH, FROSTBURG STATE UNIVERSITY 
MONIQUE REECE 
LINDA KOFFEL, HOUSTON COMMUNITY COLLEGE 
BETHANN TALSMA, DAVENPORT UNIVERSITY AND GRAND RAPIDS COMMUNITY COLLEGE 
JAMES C. HYATT,  UNIVERSITY OF THE CUMBERLANDS

——————————————————————————————————————————————————————————————————————————————————————————



#### PDF page 100 — 2,651 extracted characters

88
Chapter 3 Competing in the Global Marketplace
1983. Within weeks, Schlater’s store in Canada reached higher sales than his previous store in Ohio had
ever attained. However, it was not an easy start. Schlater had to identify the international suppliers and
get them approved to sell their products to Domino's. This shows one of the challenges that
organizations face when entering new global markets. To meet quality standards designed to protect a
brand, companies must undertake an extensive review of potential new suppliers to ensure consistent
product quality. By 2007, Schlater and a partner unified all of the franchises under one corporate
umbrella, and Schlater is now president of Domino's of Canada, Ltd., which operates more than 440
stores located in every province, as well as the Yukon and Northwest Territories.
Exhibit 3.2
Domino’s store. (Credit: Mr. Blue Mau Mau/ Flickr/ Attribution 2.0 Generic (CC BY 2.0))
Such an impressive career path might seem like luck to some, but Sch

#### PDF page 744 — 89 extracted characters

732
Index
This OpenStax book is available for free at http://cnx.org/content/col25734/1.7

——————————————————————————————————————————————————————————————————————————————————————————



## 3. Detect obvious extraction issues

The checks flag, rather than discard, pages that are empty or unusually short, contain replacement characters or control characters, or have a very low letter ratio. These flags are retained in each page record.

In [5]:
def quality_assessment(text: str) -> dict[str, Any]:
    compact = re.sub(r'\s+', '', text)
    flags: list[str] = []
    if not compact:
        flags.append('empty_text')
    elif len(compact) < 40:
        flags.append('very_short_text')
    replacement_count = text.count('�')
    if replacement_count:
        flags.append('replacement_characters')
    control_count = sum(ord(char) < 32 and char not in '\n\r\t' for char in text)
    if control_count:
        flags.append('unexpected_control_characters')
    letter_ratio = sum(char.isalpha() for char in compact) / len(compact) if compact else 0.0
    if compact and letter_ratio < 0.15:
        flags.append('low_letter_ratio')
    return {
        'flags': flags,
        'character_count': len(text),
        'word_count': len(re.findall(r'\b\w+\b', text)),
        'replacement_character_count': replacement_count,
        'control_character_count': control_count,
        'letter_ratio': round(letter_ratio, 4),
    }

raw_quality = {page: quality_assessment(text) for page, text in raw_page_text.items()}
flag_counts = Counter(flag for result in raw_quality.values() for flag in result['flags'])
flagged_pages = [{'page_number': page, **result} for page, result in raw_quality.items() if result['flags']]
display(JSON({'flag_counts': dict(flag_counts), 'flagged_page_count': len(flagged_pages), 'first_flagged_pages': flagged_pages[:10]}))

<IPython.core.display.JSON object>

## 4. Apply conservative cleanup

Cleanup normalizes Unicode, removes invisible soft-hyphen and zero-width artifacts, repairs line-break hyphenation between letters, and normalizes whitespace. It does **not** delete headings, tables, page content, citations, or flagged pages.

In [6]:
def clean_extracted_text(text: str) -> str:
    cleaned = unicodedata.normalize('NFKC', text)
    cleaned = cleaned.replace('\u00ad', '').replace('\u200b', '')
    cleaned = re.sub(r'(?<=\w)-\s*\n\s*(?=\w)', '', cleaned)
    cleaned = re.sub(r'[\t\f\v]+', ' ', cleaned)
    cleaned = re.sub(r'[ \u00a0]+', ' ', cleaned)
    cleaned = re.sub(r' *\n *', '\n', cleaned)
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
    return cleaned.strip()

cleaned_page_text = {page: clean_extracted_text(text) for page, text in raw_page_text.items()}
for page_number in sample_page_numbers[:2]:
    display(Markdown(f'#### Cleaned PDF page {page_number}'))
    print(cleaned_page_text[page_number][:1500] if cleaned_page_text[page_number] else '[No extractable text on this page.]')
    print('\n' + '—' * 90 + '\n')

#### Cleaned PDF page 1

[No extractable text on this page.]

——————————————————————————————————————————————————————————————————————————————————————————



#### Cleaned PDF page 3

Introduction to Business
SENIOR CONTRIBUTING AUTHORS
LAWRENCE J. GITMAN, SAN DIEGO STATE UNIVERSITY - EMERITUS
CARL MCDANIEL, UNIVERSITY OF TEXAS, ARLINGTON
AMIT SHAH, FROSTBURG STATE UNIVERSITY
MONIQUE REECE
LINDA KOFFEL, HOUSTON COMMUNITY COLLEGE
BETHANN TALSMA, DAVENPORT UNIVERSITY AND GRAND RAPIDS COMMUNITY COLLEGE
JAMES C. HYATT, UNIVERSITY OF THE CUMBERLANDS

——————————————————————————————————————————————————————————————————————————————————————————



## 5. Preserve chapter and section context in structured page records

Context is detected from explicit chapter and numbered-section headings and then carried forward to later pages until a new heading appears. Unknown front-matter context remains `null`, which is more trustworthy than fabricating a label.

In [7]:
CHAPTER_RE = re.compile(r'^\s*Chapter\s+(\d+)(?:\s*[:\-–—]?\s*(.*))?\s*$', re.IGNORECASE)
SECTION_RE = re.compile(r'^\s*(\d+\.\d+)\s+(.{2,160})\s*$')

def detected_context(text: str) -> dict[str, Any]:
    chapter = None
    section = None
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for index, line in enumerate(lines):
        chapter_match = CHAPTER_RE.match(line)
        if chapter_match:
            title = chapter_match.group(2).strip() if chapter_match.group(2) else (lines[index + 1] if index + 1 < len(lines) else None)
            chapter = {'number': int(chapter_match.group(1)), 'title': title or None}
        section_match = SECTION_RE.match(line)
        if section_match:
            section = {'number': section_match.group(1), 'title': section_match.group(2)}
    return {'chapter': chapter, 'section': section}

page_records: list[dict[str, Any]] = []
active_chapter = None
active_section = None
for page_number in range(1, page_count + 1):
    raw_text = raw_page_text[page_number]
    cleaned_text = cleaned_page_text[page_number]
    context = detected_context(cleaned_text)
    if context['chapter'] is not None:
        active_chapter = context['chapter']
        active_section = None
    if context['section'] is not None:
        active_section = context['section']
    page_records.append({
        'page_number': page_number,
        'chapter': active_chapter,
        'section': active_section,
        'source': SOURCE,
        'extracted_text': raw_text,
        'cleaned_text': cleaned_text,
        'quality_flags': raw_quality[page_number]['flags'],
        'extraction_metrics': {key: value for key, value in raw_quality[page_number].items() if key != 'flags'},
    })

required_fields = {'page_number', 'chapter', 'section', 'source', 'extracted_text', 'cleaned_text', 'quality_flags'}
assert len(page_records) == page_count
assert all(required_fields <= record.keys() for record in page_records)
sample_index = min(2, len(page_records) - 1)
sample_record = {**page_records[sample_index], 'extracted_text': page_records[sample_index]['extracted_text'][:500], 'cleaned_text': page_records[sample_index]['cleaned_text'][:500]}
display(JSON({'required_fields': sorted(required_fields), 'record_count': len(page_records), 'sample_record': sample_record}))

<IPython.core.display.JSON object>

## 6. Save processed output and validate completeness

The JSONL file contains one auditable structured page record per PDF page. The summary records source information, page coverage, quality-flag counts, and the Phase 02 scope boundary.

In [8]:
records_path = PROCESSED_DIR / 'introduction_to_business_page_records.jsonl'
summary_path = PROCESSED_DIR / 'introduction_to_business_ingestion_summary.json'

with records_path.open('w', encoding='utf-8') as handle:
    for record in page_records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

summary = {
    'source': SOURCE,
    'pdf_page_count': page_count,
    'record_count': len(page_records),
    'pages_with_text': sum(bool(record['cleaned_text']) for record in page_records),
    'quality_flag_counts': dict(flag_counts),
    'record_fields': sorted(required_fields | {'extraction_metrics'}),
    'phase_scope': 'PDF extraction and structured page records only; no embeddings, vector store, retrieval, reranking, LLM, or LangGraph.',
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

with records_path.open(encoding='utf-8') as handle:
    persisted_records = [json.loads(line) for line in handle if line.strip()]
assert len(persisted_records) == page_count
assert all(required_fields <= record.keys() for record in persisted_records)
assert [record['page_number'] for record in persisted_records] == list(range(1, page_count + 1))

display(JSON({
    'processed_records': str(records_path.relative_to(PROJECT_ROOT)),
    'summary': str(summary_path.relative_to(PROJECT_ROOT)),
    'record_count': len(persisted_records),
    'pages_with_text': summary['pages_with_text'],
    'quality_flag_counts': summary['quality_flag_counts'],
    'validation': 'passed',
}))

<IPython.core.display.JSON object>

## Phase 02 result

The processed page records retain the original extraction, conservative cleaned text, source provenance, page number, chapter and section context when detectable, and quality flags. Later phases must not begin until explicitly requested and must respect the documented OpenStax AI-use limitation.